In [1]:
!pip install facenet-pytorch
!pip install torchmetrics
!pip install tqdm


  Using cached facenet_pytorch-2.6.0-py3-none-any.whl.metadata (12 kB)
  Using cached numpy-1.26.4-cp314-cp314-linux_x86_64.whl
  Using cached pillow-10.2.0.tar.gz (46.2 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      Traceback (most recent call last):
        File "/home/sabyasachi19/PycharmProjects/EE656/.venv/lib/python3.14/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 353, in <module>
          main()
          ~~~~^^
        File "/home/sabyasachi19/PycharmProjects/EE656/.venv/lib/python3.14/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 335, in main
          json_out['return_val'] = hook(**hook_input['kwargs'])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "/home/sabyasachi19/PycharmProjects/

In [2]:
import os
import random
import numpy as np

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import transforms
from torchvision import models
from torchvision.datasets import ImageFolder

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from tqdm import tqdm

In [3]:
import matplotlib.pyplot as plt

In [4]:
from torchvision.datasets import ImageFolder

TRAIN_PATH = "/home/sabyasachi19/PycharmProjects/EE656/ishita_version/AffectNet_datset/archive (3)/Train"
TEST_PATH  = "/home/sabyasachi19/PycharmProjects/EE656/ishita_version/AffectNet_datset/archive (3)/Test"

train_imagefolder = ImageFolder(TRAIN_PATH)
test_imagefolder  = ImageFolder(TEST_PATH)

IMG_SIZE = 112
BATCH_SIZE = 32

In [5]:
from torchvision.datasets import ImageFolder

train_dataset_raw = ImageFolder(TRAIN_PATH)

print("Classes:", train_dataset_raw.classes)
print("Num Classes:", len(train_dataset_raw.classes))
print("Total Train Images:", len(train_dataset_raw))

Classes: ['anger', 'contempt', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Num Classes: 8
Total Train Images: 16108


In [6]:
"""for cls, count in zip(dataset.classes,
                      [sum(1 for _, y in dataset.samples if y == i)
                       for i in range(len(dataset.classes))]):
    print(cls, count)"""

'for cls, count in zip(dataset.classes,\n                      [sum(1 for _, y in dataset.samples if y == i)\n                       for i in range(len(dataset.classes))]):\n    print(cls, count)'

In [7]:
train_imagefolder = ImageFolder(TRAIN_PATH)
test_imagefolder  = ImageFolder(TEST_PATH)

train_samples = train_imagefolder.samples
test_samples  = test_imagefolder.samples

train_labels = [label for _, label in train_samples]
test_labels  = [label for _, label in test_samples]

In [8]:
from torch.utils.data import Dataset
from torchvision.datasets import ImageFolder
from PIL import Image
import random


class PairedFERDataset(Dataset):

    def __init__(self, samples, transform=None):

        self.samples = samples
        self.transform = transform

        self.label_to_indices = {}

        for idx, (_, label) in enumerate(self.samples):

            self.label_to_indices.setdefault(label, []).append(idx)

        self.valid_indices = []

        for idx, (_, label) in enumerate(self.samples):

            if len(self.label_to_indices[label]) >= 2:
                self.valid_indices.append(idx)

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):

        idx = self.valid_indices[idx]

        img1_path, label = self.samples[idx]

        candidate_indices = self.label_to_indices[label]

        partner_idx = idx

        while partner_idx == idx:
            partner_idx = random.choice(candidate_indices)

        img2_path, _ = self.samples[partner_idx]

        img1 = Image.open(img1_path).convert("RGB")
        img2 = Image.open(img2_path).convert("RGB")

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, label

In [9]:
folds = [
    {
        "train": train_samples,
        "val": test_samples
    }
]

In [10]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [11]:
from torchvision import transforms
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode
import random


class FERAugmentation:
    def __init__(self):

        self.angles = [-15, -10, -5, 0, 5, 10, 15]

        self.normalize = transforms.Normalize(
            mean=[0.5, 0.5, 0.5],
            std=[0.5, 0.5, 0.5]
        )

    def __call__(self, img):

        angle = random.choice(self.angles)
        img = TF.rotate(
            img,
            angle=angle,
            interpolation=InterpolationMode.BILINEAR,
            fill=(128, 128, 128)
        )
        if random.random() < 0.5:
            img = TF.hflip(img)

        img = TF.resize(img, (112, 112))
        img = TF.to_tensor(img)
        img = self.normalize(img)

        return img


transform = FERAugmentation()

In [12]:
CURRENT_FOLD = 0

train_dataset = PairedFERDataset(
    folds[CURRENT_FOLD]["train"],
    transform=transform
)

val_dataset = PairedFERDataset(
    folds[CURRENT_FOLD]["val"],
    transform=transform
)

In [13]:
from torchvision.datasets import ImageFolder
from torch.utils.data import Subset
import numpy as np

dataset = ImageFolder("/home/sabyasachi19/PycharmProjects/EE656/ishita_version/AffectNet_datset/archive (3)/Train")

all_labels = np.array(dataset.targets)

folds = []

for fold_id, (train_idx, val_idx) in enumerate(
        skf.split(np.arange(len(all_labels)), all_labels)
):

    train_samples = Subset(dataset, train_idx)
    val_samples   = Subset(dataset, val_idx)

    folds.append({
        "train": train_samples,
        "val": val_samples
    })

In [14]:

from torch.utils.data import Dataset
from PIL import Image
import random


In [15]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [16]:
import torch
import torch.nn as nn
from torchvision import models


class ExpressionEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        backbone = models.resnet18(weights='IMAGENET1K_V1')
        backbone.conv1 = nn.Conv2d(3,64,kernel_size=7,stride=2, padding=3,bias=False)

        self.feature_extractor = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4
        )

        self.avgpool = nn.AdaptiveAvgPool2d(1)

        self.projection = nn.Linear(512, 64)

    def forward(self, x):

        fmap = self.feature_extractor(x)

        pooled = self.avgpool(fmap)

        pooled = pooled.flatten(1)

        z = self.projection(pooled)

        return z, pooled, fmap

In [17]:
class IdentityEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        backbone = models.resnet18(weights='IMAGENET1K_V1')

        backbone.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False
        )

        self.feature_extractor = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4
        )

        self.avgpool = nn.AdaptiveAvgPool2d(1)

        self.projection = nn.Linear(512,64)

    def forward(self,x):

        fmap = self.feature_extractor(x)

        pooled = self.avgpool(fmap)

        pooled = pooled.flatten(1)

        z = self.projection(pooled)

        return z, pooled, fmap

In [18]:
class GlobalStatisticsNetwork(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(576, 512),
            nn.ELU(),

            nn.Linear(512, 256),
            nn.ELU(),

            nn.Linear(256, 1)
        )

    def forward(self, pooled_feat, z):

        x = torch.cat(
            [pooled_feat, z],
            dim=1
        )

        return self.net(x)

In [19]:
class LocalStatisticsNetwork(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Conv2d(
                576,
                512,
                kernel_size=1
            ),

            nn.ELU(),

            nn.Conv2d(
                512,
                1,
                kernel_size=1
            )
        )

    def forward(self, fmap, z):

        B,C,H,W = fmap.shape

        z = z.unsqueeze(-1).unsqueeze(-1)

        z = z.expand(-1,-1,H,W)

        x = torch.cat(
            [fmap, z],
            dim=1
        )

        score_map = self.net(x)

        return score_map

In [20]:
import math
import torch
import torch.nn as nn


class StableMINE(nn.Module):

    def __init__(self, stats_network):

        super().__init__()

        self.stats_network = stats_network

        self.register_buffer(
            "ema_denominator",
            torch.tensor(1.0)
        )

        self.ema_alpha = 0.01

    def forward(self,
                image_features,
                latent_codes):

        N = image_features.size(0)

        joint_scores = self.stats_network(
            image_features,
            latent_codes
        )

        joint_mean = joint_scores.mean()

        x_expand = image_features.unsqueeze(1)
        x_expand = x_expand.expand(
            N,
            N,
            image_features.size(1)
        )

        z_expand = latent_codes.unsqueeze(0)
        z_expand = z_expand.expand(
            N,
            N,
            latent_codes.size(1)
        )

        all_pairs_x = x_expand.reshape(
            N * N,
            image_features.size(1)
        )

        all_pairs_z = z_expand.reshape(
            N * N,
            latent_codes.size(1)
        )

        all_scores = self.stats_network(
            all_pairs_x,
            all_pairs_z
        )

        all_scores = all_scores.squeeze()
        log_mean_exp = (
            torch.logsumexp(
                all_scores,
                dim=0
            )
            -
            math.log(N * N)
        )

        mi_estimate = (
            joint_mean
            -
            log_mean_exp
        )

        loss = -mi_estimate

        return loss, mi_estimate.detach()

In [21]:
import math
import torch
import torch.nn as nn


class LocalMINE(nn.Module):

    def __init__(self, stats_network):

        super().__init__()

        self.stats_network = stats_network

    def forward(self,
                fmap,
                latent_codes):

        B,C,H,W = fmap.shape

        joint_scores = self.stats_network(
            fmap,
            latent_codes
        )

        joint_scores = joint_scores.sum(dim=[2,3])

        joint_mean = joint_scores.mean()

        fmap_expand = fmap.unsqueeze(1)

        fmap_expand = fmap_expand.expand(B,B,C,H,W)

        z_expand = latent_codes.unsqueeze(0)

        z_expand = z_expand.expand(B,B,latent_codes.size(1))

        all_fmaps = fmap_expand.reshape(B * B, C,H,W
        )

        all_z = z_expand.reshape(
            B * B,
            latent_codes.size(1)
        )

        all_scores = self.stats_network(
            all_fmaps,
            all_z
        )

        all_scores = all_scores.sum(dim=[2,3])

        all_scores = all_scores.squeeze()

        log_mean_exp = (
            torch.logsumexp(
                all_scores,
                dim=0
            )
            -
            math.log(B * B)
        )

        mi_estimate = (joint_mean-log_mean_exp
        )

        loss = -mi_estimate

        return loss, mi_estimate.detach()

In [22]:
class Stage1Loss(nn.Module):

    def __init__(self, global_mine, local_mine, delta=0.1):

        super().__init__()

        self.global_mine = global_mine
        self.local_mine = local_mine

        self.delta = delta

        self.l1_loss = nn.L1Loss()

    def forward(self,
                pooled_M,
                pooled_N,
                fmap_M,
                fmap_N,
                E_M,
                E_N):

        global_loss_M, global_mi_M = self.global_mine(pooled_M, E_N)

        global_loss_N, global_mi_N = self.global_mine(pooled_N, E_M)

        local_loss_M, local_mi_M = self.local_mine(fmap_M, E_N)

        local_loss_N, local_mi_N = self.local_mine(fmap_N, E_M)

        global_loss = 0.5 * (global_loss_M + global_loss_N)

        local_loss = 1.0 * (local_loss_M + local_loss_N)

        l1 = self.l1_loss(E_M, E_N)

        total_loss = global_loss + local_loss + self.delta * l1

        global_mi = 0.5 * (global_mi_M + global_mi_N)

        local_mi = 1.0 * (local_mi_M + local_mi_N)

        return total_loss, global_mi, local_mi, l1

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [24]:
expr_encoder = ExpressionEncoder().to(device)

global_stats_net = GlobalStatisticsNetwork().to(device)

local_stats_net = LocalStatisticsNetwork().to(device)

global_mine = StableMINE(global_stats_net).to(device)

local_mine = LocalMINE(local_stats_net).to(device)

stage1_criterion = Stage1Loss(
    global_mine,
    local_mine,
    delta=0.1
)

In [25]:
optimizer_stage1 = torch.optim.Adam(
    list(expr_encoder.parameters()) +
    list(global_stats_net.parameters()) +
    list(local_stats_net.parameters()),
    lr=1e-4
)

In [26]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor()
])

In [27]:
NUM_EPOCHS_STAGE1 = 40

for epoch in range(NUM_EPOCHS_STAGE1):

    expr_encoder.train()

    epoch_loss = 0
    epoch_global_mi = 0
    epoch_local_mi = 0
    epoch_l1 = 0

    for batch in train_loader:

        M, N, labels = batch

        M = M.to(device)
        N = N.to(device)
        labels = labels.to(device)

        optimizer_stage1.zero_grad(set_to_none=True)

        E_M, pooled_M, fmap_M = expr_encoder(M)
        E_N, pooled_N, fmap_N = expr_encoder(N)

        loss, global_mi, local_mi, l1 = stage1_criterion(
            pooled_M, pooled_N,
            fmap_M, fmap_N,
            E_M, E_N
        )

        loss.backward()
        optimizer_stage1.step()

        epoch_loss += loss.item()
        epoch_global_mi += global_mi.item()
        epoch_local_mi += local_mi.item()
        epoch_l1 += l1.item()

    avg_loss = epoch_loss / len(train_loader)
    avg_global_mi = epoch_global_mi / len(train_loader)
    avg_local_mi = epoch_local_mi / len(train_loader)
    avg_l1 = epoch_l1 / len(train_loader)

    print(
        f"Epoch {epoch+1:03d} | "
        f"GMI={avg_global_mi:.4f} | "
        f"LMI={avg_local_mi:.4f} | "
        f"L1={avg_l1:.4f} | "
        f"Loss={avg_loss:.4f}"
    )

Epoch 001 | GMI=-0.0003 | LMI=-1.0231 | L1=0.4668 | Loss=1.0700
Epoch 002 | GMI=0.0000 | LMI=-0.0424 | L1=0.2973 | Loss=0.0721
Epoch 003 | GMI=0.0007 | LMI=-0.0098 | L1=0.2027 | Loss=0.0293
Epoch 004 | GMI=0.2110 | LMI=0.3664 | L1=0.2139 | Loss=-0.5560
Epoch 005 | GMI=0.4620 | LMI=0.9151 | L1=0.1583 | Loss=-1.3613
Epoch 006 | GMI=0.5055 | LMI=0.9999 | L1=0.1409 | Loss=-1.4914
Epoch 007 | GMI=0.5455 | LMI=1.0798 | L1=0.1449 | Loss=-1.6109
Epoch 008 | GMI=0.5778 | LMI=1.1589 | L1=0.1657 | Loss=-1.7201
Epoch 009 | GMI=0.6080 | LMI=1.2140 | L1=0.1767 | Loss=-1.8044
Epoch 010 | GMI=0.6434 | LMI=1.2813 | L1=0.1819 | Loss=-1.9065
Epoch 011 | GMI=0.6797 | LMI=1.3557 | L1=0.1801 | Loss=-2.0173
Epoch 012 | GMI=0.6979 | LMI=1.3909 | L1=0.1830 | Loss=-2.0705
Epoch 013 | GMI=0.7291 | LMI=1.4519 | L1=0.1957 | Loss=-2.1614
Epoch 014 | GMI=0.7622 | LMI=1.5159 | L1=0.1975 | Loss=-2.2584
Epoch 015 | GMI=0.7944 | LMI=1.5760 | L1=0.1949 | Loss=-2.3509
Epoch 016 | GMI=0.8218 | LMI=1.6303 | L1=0.1947 | Loss

In [28]:
torch.save(expr_encoder.state_dict(), "/home/sabyasachi19/PycharmProjects/EE656/ishita_version/saves/checkpoint_affecnet")

In [29]:
class GlobalStatisticsNetworkID(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512 + 128, 512), nn.ELU(),
            nn.Linear(512, 256), nn.ELU(),
            nn.Linear(256, 1),
        )

    def forward(self, pooled_feat, t):
        x = torch.cat([pooled_feat, t], dim=1)
        return self.net(x)


class LocalStatisticsNetworkID(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(512 + 128, 512, kernel_size=1), nn.ELU(),
            nn.Conv2d(512, 1, kernel_size=1),
        )

    def forward(self, fmap, t):
        B, C, H, W = fmap.shape
        t = t.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, H, W)
        x = torch.cat([fmap, t], dim=1)
        return self.net(x)

In [30]:
global_mine_id = StableMINE(GlobalStatisticsNetworkID()).to(device)
local_mine_id = LocalMINE(LocalStatisticsNetworkID()).to(device)

In [31]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64 + 64, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, e, i):
        x = torch.cat([e, i], dim=1)
        return self.net(x)

In [32]:
def shuffle(x):
    idx = torch.randperm(x.size(0), device=x.device)
    return x[idx]

def discriminator_loss(Discriminator,e,i):
  i2=shuffle(i)
  out_pos=Discriminator(e,i)
  out_neg=Discriminator(e,i2)
  y_pos = torch.zeros_like(out_pos)
  y_neg = torch.ones_like(out_neg)

  return F.binary_cross_entropy_with_logits(out_pos, y_pos) + \
           F.binary_cross_entropy_with_logits(out_neg, y_neg)

def encoder_loss(Discriminator,e,i):
  fake_logits=Discriminator(e,i)
  y = torch.ones_like(fake_logits)
  return F.binary_cross_entropy_with_logits(fake_logits, y)

@torch.no_grad()
def discriminator_accuracy(Discriminator, e, i):
    i2 = shuffle(i)
    fake_logits = Discriminator(e, i)
    real_logits = Discriminator(e, i2)

    fake_correct= torch.sigmoid(Discriminator(e, i)) < 0.5
    real_correct = torch.sigmoid(Discriminator(e, i2)) >= 0.5

    return (fake_correct.float().sum() + real_correct.float().sum()) / (fake_correct.numel() + real_correct.numel())





In [33]:
class Stage2MILoss(nn.Module):
    def __init__(self, global_mine_id, local_mine_id):
        super().__init__()
        self.global_mine_id = global_mine_id
        self.local_mine_id = local_mine_id

    def forward(self, pooled_M, pooled_N, fmap_M, fmap_N, T_M, T_N):
        global_loss_M, global_mi_M = self.global_mine_id(pooled_M, T_M)
        global_loss_N, global_mi_N = self.global_mine_id(pooled_N, T_N)
        local_loss_M, local_mi_M = self.local_mine_id(fmap_M, T_M)
        local_loss_N, local_mi_N = self.local_mine_id(fmap_N, T_N)

        global_loss = 0.5 * (global_loss_M + global_loss_N)
        local_loss = 0.5 * (local_loss_M + local_loss_N)
        total_loss = global_loss + local_loss

        global_mi = 0.5 * (global_mi_M + global_mi_N)
        local_mi = 0.5 * (local_mi_M + local_mi_N)
        return total_loss, global_mi, local_mi

In [34]:
identity_enc = IdentityEncoder().to(device)
disc = Discriminator().to(device)
mi_loss_fn = Stage2MILoss(global_mine_id, local_mine_id).to(device)

for p in expr_encoder.parameters():
    p.requires_grad = False
expr_encoder.eval()

optimizer_g = torch.optim.Adam(
    list(identity_enc.parameters()) +
    list(global_mine_id.parameters()) +
    list(local_mine_id.parameters()),
    lr=1e-4
)

optimizer_d = torch.optim.Adam(disc.parameters(), lr=1e-4)

adv_w = 0.025
epochs = 30


for ep in range(epochs):
    identity_enc.train()

    mi_sum, acc_sum, steps = 0.0, 0.0, 0

    for M, N, _ in train_loader:
        M, N = M.to(device), N.to(device)

        with torch.no_grad():
            EM, _, _ = expr_encoder(M)
            EN, _, _ = expr_encoder(N)

        IM, pIM, fIM = identity_enc(M)
        IN, pIN, fIN = identity_enc(N)

        d_loss = (
            discriminator_loss(disc, EM, IM.detach()) +
            discriminator_loss(disc, EN, IN.detach())
        )

        optimizer_d.zero_grad()
        d_loss.backward()
        optimizer_d.step()

        TM = torch.cat([EM, IM], dim=1)
        TN = torch.cat([EN, IN], dim=1)

        mi_loss, g_mi, l_mi = mi_loss_fn(
            pIM, pIN, fIM, fIN, TM, TN
        )

        fool = (
            encoder_loss(disc, EM, IM) +
            encoder_loss(disc, EN, IN)
        )

        g_loss = mi_loss + adv_w * fool

        optimizer_g.zero_grad()
        g_loss.backward()
        optimizer_g.step()

        with torch.no_grad():
            acc_val = (
                discriminator_accuracy(disc, EM, IM) +
                discriminator_accuracy(disc, EN, IN)
            ) / 2

        mi_sum += (g_mi + l_mi).item()
        acc_sum += acc_val
        steps += 1

    print(f"Epoch {ep+1:03d} | MI={mi_sum/steps:.4f} | disc_acc={acc_sum/steps:.3f}")

KeyboardInterrupt: 

In [ ]:
torch.save(identity_enc.state_dict(), "/home/sabyasachi19/PycharmProjects/EE656/ishita_version/saves/AFFECTNETidentity_encoder_stage2.pt")
torch.save(disc.state_dict(), "/home/sabyasachi19/PycharmProjects/EE656/ishita_version/saves/savesAFFECTNETdiscriminator_stage2.pt")

In [ ]:
class FERClassifier(nn.Module):
    def __init__(self, embedding_dim=64, num_classes=7, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, e):
        return self.net(e)


classifier = FERClassifier(num_classes=len(dataset.classes)).to(device)
classifier_optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

NUM_EPOCHS_CLASSIFIER = 30
for epoch in range(NUM_EPOCHS_CLASSIFIER):
    classifier.train()
    epoch_loss, correct, total = 0.0, 0, 0
    for M, N, label in train_loader:
        M, label = M.to(device), label.to(device)
        with torch.no_grad():
            E_M, _, _ = expr_encoder(M)
        logits = classifier(E_M)
        loss = criterion(logits, label)
        classifier_optimizer.zero_grad(); loss.backward(); classifier_optimizer.step()
        epoch_loss += loss.item()
        correct += (logits.argmax(dim=1) == label).sum().item()
        total += label.size(0)
    train_acc = correct / total

    classifier.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for M, N, label in val_loader:
            M, label = M.to(device), label.to(device)
            E_M, _, _ = expr_encoder(M)
            val_correct += (classifier(E_M).argmax(dim=1) == label).sum().item()
            val_total += label.size(0)
    val_acc = val_correct / val_total
    print(f"Epoch {epoch+1:03d} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

In [ ]:
class PairStatisticsNetwork(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim * 2, 128), nn.ELU(),
            nn.Linear(128, 64), nn.ELU(),
            nn.Linear(64, 1),
        )

    def forward(self, a, b):
        return self.net(torch.cat([a, b], dim=1))


def compute_mig(e_m, e_n, i_m, device, train_steps=300, lr=1e-3):
    mine_ee = StableMINE(PairStatisticsNetwork(64)).to(device)
    mine_ei = StableMINE(PairStatisticsNetwork(64)).to(device)
    opt_ee = torch.optim.Adam(mine_ee.parameters(), lr=lr)
    opt_ei = torch.optim.Adam(mine_ei.parameters(), lr=lr)
    n = e_m.size(0)
    batch_size = min(64, n)

    def _train_one(mine_net, opt, x, y):
        best = -float("inf")
        for _ in range(train_steps):
            idx = torch.randperm(n, device=device)[:batch_size]
            loss, mi = mine_net(x[idx], y[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            best = max(best, mi.item())
        return best

    i_ee = _train_one(mine_ee, opt_ee, e_m, e_n)
    i_ei = _train_one(mine_ei, opt_ei, e_m, i_m)
    return {"I_E_E": i_ee, "I_E_I": i_ei, "MIG": i_ee - i_ei}


eval_M, eval_N = [], []
for M, N, _ in val_loader:
    eval_M.append(M); eval_N.append(N)
eval_M, eval_N = torch.cat(eval_M).to(device), torch.cat(eval_N).to(device)

with torch.no_grad():
    eval_E_M, _, _ = expr_encoder(eval_M)
    eval_E_N, _, _ = expr_encoder(eval_N)
    eval_I_M, _, _ = identity_enc(eval_M)

print(compute_mig(eval_E_M, eval_E_N, eval_I_M, device))

In [ ]:
!pip install seaborn

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

classifier.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for M, N, label in val_loader:
        M = M.to(device)
        E_M, _, _ = expr_encoder(M)
        all_preds.append(classifier(E_M).argmax(dim=1).cpu())
        all_labels.append(label)
all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(dataset.classes))), normalize="true")
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt=".2f", xticklabels=dataset.classes, yticklabels=dataset.classes, cmap="Greens")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
plt.show()

print(classification_report(all_labels, all_preds, labels=list(range(len(dataset.classes))),
                             target_names=dataset.classes, zero_division=0))

In [ ]:
def denormalize(img_tensor):
    img = (img_tensor * 0.5 + 0.5).clamp(0, 1)
    return img.permute(1, 2, 0).cpu().numpy()


bank_images, bank_E, bank_I = [], [], []
expr_encoder.eval(); identity_enc.eval()
with torch.no_grad():
    for M, N, label in val_loader:
        M = M.to(device)
        E_M, _, _ = expr_encoder(M)
        I_M, _, _ = identity_enc(M)
        bank_images.append(M.cpu()); bank_E.append(E_M.cpu()); bank_I.append(I_M.cpu())

bank_images = torch.cat(bank_images)
bank_E = F.normalize(torch.cat(bank_E), dim=1)
bank_I = F.normalize(torch.cat(bank_I), dim=1)


def cosine_retrieval(query_idx, bank, top_k=5):
    sims = bank @ bank[query_idx]
    sims[query_idx] = -1.0
    top_sims, top_idx = sims.topk(top_k)
    return top_idx.numpy(), top_sims.numpy()


query_idx = 0
e_nn, e_sims = cosine_retrieval(query_idx, bank_E)
i_nn, i_sims = cosine_retrieval(query_idx, bank_I)

fig, axes = plt.subplots(2, 6, figsize=(16, 5))
axes[0, 0].imshow(denormalize(bank_images[query_idx])); axes[0, 0].set_title("Query"); axes[0, 0].axis("off")
for j, idx in enumerate(e_nn):
    axes[0, j+1].imshow(denormalize(bank_images[idx]))
    axes[0, j+1].set_title(f"E-NN {j+1}\nsim={e_sims[j]:.2f}"); axes[0, j+1].axis("off")
axes[1, 0].imshow(denormalize(bank_images[query_idx])); axes[1, 0].set_title("Query"); axes[1, 0].axis("off")
for j, idx in enumerate(i_nn):
    axes[1, j+1].imshow(denormalize(bank_images[idx]))
    axes[1, j+1].set_title(f"I-NN {j+1}\nsim={i_sims[j]:.2f}"); axes[1, j+1].axis("off")
plt.suptitle("Top: retrieved by Expression (E)  |  Bottom: retrieved by Identity (I)")
plt.tight_layout(); plt.show()